In [32]:
import os
import ast
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import torch
from lingua import LanguageDetectorBuilder
from sentence_transformers import SentenceTransformer

In [33]:
#Taking a sample because dealing with 99 million rows of data at once is computationally heavy
#train: 250k AI + 250k not AI picked at random from the whole file
#test: 200k labeled posts at random with the real AI rate, never overlapping train

source = pq.ParquetFile("Data/2026-09.parquet")
output = "Data/sample_500k.parquet"
test_output = "Data/test_natural_200k.parquet"

n_per_class = 250_000
n_test = 200_000

labels = pq.read_table("Data/2026-09.parquet", columns=["is_ai_generated"])["is_ai_generated"].to_numpy(zero_copy_only=False)
n_ai = int(np.sum(labels == 1))
n_human = int(np.sum(labels == 0))
print(f"AI: {n_ai:,}  Not AI: {n_human:,}")

p_test = 1.1 * n_test / (n_ai + n_human)
p_ai = min(1.0, 1.1 * n_per_class / (n_ai * (1 - p_test)))
p_human = min(1.0, 1.1 * n_per_class / (n_human * (1 - p_test)))

rng = np.random.default_rng(42)
train_parts, test_parts = [], []

for i, batch in enumerate(source.iter_batches(batch_size=1_000_000)):
    y = batch.column("is_ai_generated").to_numpy(zero_copy_only=False).astype(float)
    u = rng.random(len(y))
    labeled = ~np.isnan(y)

    test_mask = labeled & (u < p_test)
    u_train = (u - p_test) / (1 - p_test)
    train_mask = ~test_mask & (((y == 1) & (u_train < p_ai)) | ((y == 0) & (u_train < p_human)))

    train_parts.append(batch.filter(pa.array(train_mask)))
    test_parts.append(batch.filter(pa.array(test_mask)))

    if i % 10 == 0:
        print(f"Batches read: {i + 1}")

train = pa.Table.from_batches(train_parts)
test = pa.Table.from_batches(test_parts)

y_train = train["is_ai_generated"].to_numpy(zero_copy_only=False).astype(float)
keep_ai = rng.choice(np.flatnonzero(y_train == 1), size=n_per_class, replace=False)
keep_human = rng.choice(np.flatnonzero(y_train == 0), size=n_per_class, replace=False)
train = train.take(rng.permutation(np.concatenate([keep_ai, keep_human])))

test = test.take(rng.choice(test.num_rows, size=n_test, replace=False))

pq.write_table(train, output)
pq.write_table(test, test_output)

y_test = test["is_ai_generated"].to_numpy(zero_copy_only=False).astype(float)
print(f"Train saved: {train.num_rows:,} rows, AI rate {np.mean(y_train[np.concatenate([keep_ai, keep_human])]):.3f}")
print(f"Test saved: {test.num_rows:,} rows, AI rate {np.mean(y_test):.3f}")
print("Sample saved!")

AI: 3,379,480  Not AI: 55,496,506
Batches read: 1
Batches read: 11
Batches read: 21
Batches read: 31
Batches read: 41
Batches read: 51
Batches read: 61
Batches read: 71
Batches read: 81
Batches read: 91
Train saved: 500,000 rows, AI rate 0.500
Test saved: 200,000 rows, AI rate 0.058
Sample saved!


In [34]:
#loading 500k samples

df = pd.read_parquet('Data/sample_500k.parquet')
print("Rows:", len(df))

Rows: 500000


In [35]:
print(df['is_ai_generated'].sum()) #5000 out of 120k are ai

#looking at the dtypes for analyzing
print(df.dtypes)

print(df['language'].unique())
print(len(df[df['language'] == 'un']))

dl = df.groupby('language').mean(numeric_only=True)

stats = (df.groupby('language')['is_ai_generated']
           .agg(rate='mean', n='size', labeled='count')
           .sort_values('n', ascending=False))

print(stats[stats['labeled'] >= 30].sort_values('rate', ascending=False))

print(df.groupby('not_recommended_for_fyp')['is_ai_generated'].mean())

print(df.isna().sum())

250000
video_id                                uint64
posted_at                  datetime64[ms, UTC]
country                                    str
language                                   str
type                                       str
duration_ms                             uint32
image_count                              uint8
width                                   uint16
height                                  uint16
caption                                    str
hashtags                                object
hashtag_ids                             object
mentions                                object
on_screen_text                          object
sound_id                                uint64
is_ad                                    uint8
branded_content                            str
product_id                              uint64
is_shop_video                            uint8
seller_id                               uint64
is_ai_generated                          uint8
not_re

In [36]:
df['caption'] = df['caption'].str.strip().replace('', np.nan)

#we would like to take the un and fill it using the caption since it has much data about the language of the video
print(df['caption'].isna().sum())
print(df[(df['language'] == 'un') & (df['caption'].isna())])

detector = LanguageDetectorBuilder.from_all_languages().build()

mask = (df['language'] == 'un') & df['caption'].notna()

# Get all captions that need detection
texts = df.loc[mask, 'caption'].tolist()

# Detect languages in parallel
results = detector.compute_language_confidence_values_in_parallel(texts)

# Convert results to language codes
def get_code(result):
    if not result or result[0].value < 0.5:
        return 'un'

    code = result[0].language.iso_code_639_1
    return code.name.lower() if code else 'un'

df.loc[mask, 'language'] = [get_code(r) for r in results]

print("Remaining unknown:", (df['language'] == 'un').sum())

#NOTE ONLY AROUND 7K ENTRIES GOT FILLED

#frequency encoding for the data
freq = df['language'].value_counts(normalize=True)   # each language -> its share of rows
df['lang_freq'] = df['language'].map(freq)           # replace each row's language with that share
df = df.drop(columns=['language'])

95816
                   video_id                 posted_at country language   type  \
2       7688130262428814605 2026-09-21 23:12:15+00:00      US       un  video   
9       7687145007773502750 2026-09-19 07:29:25+00:00      US       un  video   
15      7691274249826290975 2026-09-30 10:32:28+00:00      US       un  video   
17      7681210420136365342 2026-09-03 07:40:28+00:00      US       un  video   
35      7684041441781304597 2026-09-10 22:45:20+00:00      US       un  photo   
...                     ...                       ...     ...      ...    ...   
499984  7683894775795256607 2026-09-10 13:16:21+00:00      KE       un  photo   
499987  7685108901712350477 2026-09-13 19:47:44+00:00      US       un  video   
499989  7680641196577885461 2026-09-01 18:50:42+00:00      EG       un  video   
499990  7681202956099800338 2026-09-03 07:10:38+00:00      KH       un  photo   
499994  7681017404691893522 2026-09-02 19:10:37+00:00      PK       un  video   

        duration_ms  

In [37]:
freq = df['country'].value_counts(normalize=True)
print(freq)
df['country_freq'] = df['country'].map(freq)
df.drop(columns='country', inplace = True, errors= 'ignore')

country
US     0.509334
ID     0.044536
BR     0.030002
PH     0.026206
VN     0.020902
         ...   
TO     0.000002
PW     0.000002
689    0.000002
LI     0.000002
IM     0.000002
Name: proportion, Length: 216, dtype: float64


In [38]:
print(df['branded_content'].unique())

print(df['branded_content'].value_counts())

df['is_branded'] = ((df['branded_content'] == 'paid_partnership') |
                    (df['branded_content'] == 'shop_affiliate')).astype(int)

df = df.drop(columns=['branded_content'])

<ArrowStringArray>
['', 'shop_affiliate', 'paid_partnership']
Length: 3, dtype: str
branded_content
                    473876
shop_affiliate       20471
paid_partnership      5653
Name: count, dtype: int64


In [39]:
print(df['sound_id'].unique()) #analysis done on the whole dataset its better to keep this if we were training on the whole dataset but since we took only a part of it for this month
#and its mostly 1.5 IDs for each row which gives us very little signal we drop for the full dataset we can since the ids repeate more

df.drop(columns = 'sound_id', inplace = True)

print(df['product_id'].unique())

df = df.drop(columns=['product_id', 'seller_id'])

[7686923520533596959 7671426199813392422 7688130329474829070 ...
 7670834886374214408 7690247797031521054 7689782372959472391]
[                  0 1732617405593916230 1730927783781307026 ...
 1732503011543126918 1732595083748413853 1729638192750432615]


In [40]:
df['n_hashtags'] = df['hashtags'].apply(lambda x: len(x) if x is not None else 0)
df = df.drop(columns=['hashtag_ids'])

print(df.iloc[20:30])
#hash tags and caption gives similar data we drop hashtags
df.drop(columns='hashtags',inplace=True)

               video_id                 posted_at   type  duration_ms  \
20  7682090685712928031 2026-09-05 16:35:36+00:00  video        13118   
21  7689008745556495636 2026-09-24 11:00:00+00:00  video        74535   
22  7684123119627390222 2026-09-11 04:02:23+00:00  video        12073   
23  7681977461512473869 2026-09-05 09:18:00+00:00  video        33668   
24  7687277638616141077 2026-09-19 16:03:27+00:00  video        60001   
25  7683463941640572180 2026-09-22 02:00:00+00:00  video        18367   
26  7687638276433366293 2026-09-20 15:22:54+00:00  video        30000   
27  7688408573872835861 2026-09-25 12:00:00+00:00  video        39103   
28  7691328337217686804 2026-09-30 14:02:11+00:00  video        43867   
29  7683182443939089678 2026-09-08 15:12:08+00:00  video        83067   

    image_count  width  height  \
20            0   1080    1920   
21            0   1080    1920   
22            0   1080    1920   
23            0   1080    1920   
24            0    720    

In [41]:
def parse_mentions(x):
    if isinstance(x, str):
        return ast.literal_eval(x)
    return x

df['mentions'] = df['mentions'].apply(parse_mentions)

print(df['mentions'].apply(lambda x: len(x) == 0).sum())

df['has_mentions'] = df['mentions'].apply(lambda x: int(len(x) > 0))

df.drop(columns=['mentions','video_id'], inplace= True, errors= 'ignore')

458378


In [42]:
x = pd.get_dummies(df['type'],
               prefix='type_')
print(x)

df = df.drop(columns=['type'])
df['type_video'] = x['type__video'].astype('int8')

        type__photo  type__video
0             False         True
1              True        False
2             False         True
3             False         True
4             False         True
...             ...          ...
499995        False         True
499996         True        False
499997        False         True
499998        False         True
499999        False         True

[500000 rows x 2 columns]


In [43]:
df['update_delay_min'] = (
    df['stats_updated_at'] - df['posted_at']
).dt.total_seconds() / 60
df.drop(columns=['stats_updated_at','posted_at'],inplace= True, errors='ignore')

In [44]:
print(df['on_screen_text'])

df['on_screen_text_length'] = df['on_screen_text'].apply(
    lambda texts: sum(len(t.strip()) for t in texts if isinstance(t, str))
)

df.drop(columns='on_screen_text', inplace=True)

0                                                        []
1         [Pending in Bayside Estates! 🌴An exclusive 55+...
2                                                        []
3                                                        []
4                                                        []
                                ...                        
499995                                                   []
499996                                                   []
499997                                                   []
499998                                                   []
499999                                                   []
Name: on_screen_text, Length: 500000, dtype: object


In [ ]:
model = SentenceTransformer(
    "Qwen/Qwen3-Embedding-0.6B",
    device="cuda",
    model_kwargs={"torch_dtype": torch.float16}
)
# default max length is 32768 tokens; 99% of captions are under ~270 tokens, so limit it 
model.max_seq_length = 256

# much faster option (~18x, 384 dims) 

emb_path = "Data/caption_emb_512.npy"

if os.path.exists(emb_path):
    caption_emb = np.load(emb_path)
else:
    # only embed real, unique captions (~24k are empty and many repeat)
    has_caption = df['caption'].notna().to_numpy()
    inverse, unique_caps = pd.factorize(df.loc[has_caption, 'caption'])

    unique_emb = model.encode(
        unique_caps.tolist(),
        batch_size=128,
        prompt_name="document",
        normalize_embeddings=True,
        truncate_dim=512,
        convert_to_numpy=True,
        show_progress_bar=True
    )

    # rows without a caption get a zero vector
    caption_emb = np.zeros((len(df), unique_emb.shape[1]), dtype=np.float32)
    caption_emb[has_caption] = unique_emb[inverse]
    np.save(emb_path, caption_emb)

print(caption_emb.shape)

df.drop(columns='caption', inplace= True)

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Batches:   0%|          | 0/2838 [00:00<?, ?it/s]

In [ ]:
df['engagement_rate'] = df['engagement_rate'].fillna(0.)

df = df.astype(np.float32)

print(df.dtypes)

print(df.isna().sum())

df.to_parquet('Data/Processed.parquet')
print("Saved!")

duration_ms                float32
image_count                float32
width                      float32
height                     float32
is_ad                      float32
is_shop_video              float32
is_ai_generated            float32
not_recommended_for_fyp    float32
sound_muted                float32
views                      float32
likes                      float32
comments                   float32
shares                     float32
saves                      float32
downloads                  float32
engagement_rate            float32
lang_freq                  float32
country_freq               float32
is_branded                 float32
n_hashtags                 float32
has_mentions               float32
type_video                 float32
update_delay_min           float32
on_screen_text_length      float32
dtype: object
duration_ms                0
image_count                0
width                      0
height                     0
is_ad                      0
i